# 🛢️ Oil Spill Detection — Module 2 Training
## Look-alike Discriminator & Bilge-Dump Characterization

**Pipeline position:** Module 1 → **Module 2 (this notebook)** → Module 3 → Module 4

This notebook trains the **Random Forest look-alike discriminator** on dark-patch features
extracted from Module 1 segmentation output, then applies the bilge-dump morphology filter.

| Cell | Purpose |
|------|---------|
| Cell 0 | GPU verification |
| Cell 1 | Setup: repo clone + dependencies + Hugging Face connection |
| Cell 2 | Data symlinks + Module 1 checkpoint check |
| Cell 3 | Feature extraction (12-feature matrix per dark patch) |
| Cell 4 | Random Forest training (GroupKFold CV by scene_id) |
| Cell 5 | Bilge-dump filter + session summary |

> **Runtime estimate:** Feature extraction ~10–30 min · RF training ~2–5 min · Total < 1 hr on T4 GPU

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 0 — GPU VERIFICATION  (run first, always)
# ═══════════════════════════════════════════════════════════════════════════════
import subprocess, sys, os

print("🔍 System check...")
result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                         "--format=csv,noheader"], capture_output=True, text=True)
if result.returncode == 0:
    print(f"✅ GPU : {result.stdout.strip()}")
else:
    print("⚠️  nvidia-smi not found — CPU only (RF training still works, just slower)")

import platform
print(f"✅ Python : {sys.version.split()[0]}")
print(f"✅ OS     : {platform.system()} {platform.release()}")

try:
    import torch
    print(f"✅ PyTorch: {torch.__version__}  CUDA: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"   GPU name: {torch.cuda.get_device_name(0)}")
        print(f"   VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
except ImportError:
    print("⚠️  PyTorch not found — M1 on-the-fly inference disabled (pre-computed masks will be used)")

print("\n✅ Cell 0 complete.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1 — SETUP: Repo · Dependencies · Hugging Face Connection
# ═══════════════════════════════════════════════════════════════════════════════
import subprocess, sys, os

# ── 1. Clone / update repo ───────────────────────────────────────────────────
REPO_URL  = "https://github.com/RohithSheregar/Oil-Spill-Detection-New.git"
REPO_DIR  = "/kaggle/working/repo"

if not os.path.exists(REPO_DIR):
    print("📦 Cloning repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print("🔄 Pulling latest...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f"✅ Repo: {REPO_DIR}")

# ── 2. Install dependencies ──────────────────────────────────────────────────
print("\n📦 Installing deps...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "scikit-learn", "scikit-image", "scipy", "joblib",
    "shapely", "tifffile", "pandas", "numpy", "matplotlib",
    "huggingface_hub",
], check=True)
print("✅ Dependencies ready.")

# ── 3. Load HF token ─────────────────────────────────────────────────────────
from kaggle_secrets import UserSecretsClient

HF_TOKEN   = ""
HF_REPO_ID = "RohithSheregar/oil-spill-models"   # ← same repo as Module 1

try:
    secrets   = UserSecretsClient()
    HF_TOKEN  = secrets.get_secret("HF_TOKEN")
    print(f"\n✅ Token loaded successfully.")
except Exception as e:
    print(f"\n⚠️  Could not load HF_TOKEN: {e}")
    print("   Set it in Kaggle → Add-ons → Secrets → HF_TOKEN")

# ── 4. Test HF connection ─────────────────────────────────────────────────────
if HF_TOKEN:
    print("\n🧪 Testing Hugging Face connection...")
    try:
        from huggingface_hub import HfApi
        api  = HfApi(token=HF_TOKEN)
        user = api.whoami()
        print(f"   ✅ Authenticated as: {user['name']}")
        api.create_repo(repo_id=HF_REPO_ID, exist_ok=True, private=True)
        print(f"   ✅ Repo ready: https://huggingface.co/{HF_REPO_ID}")
    except Exception as e:
        print(f"   ⚠️  HF connection failed: {e}")
        HF_TOKEN = ""
else:
    print("\n⚠️  HF_TOKEN empty — results will NOT be uploaded to Hugging Face.")

print("\n✅ Cell 1 complete.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 2 — DATA SYMLINKS + MODULE 1 CHECKPOINT CHECK
# ─────────────────────────────────────────────────────────────────────────────
# Module 2 needs:
#   (a) The same training data as Module 1  (oil + lookalike TIFFs)
#   (b) A Module 1 checkpoint for on-the-fly inference when masks are missing
#       (optional — if gt masks are available, inference is skipped)
# ─────────────────────────────────────────────────────────────────────────────
import glob, os, shutil
from pathlib import Path

INPUT_DIR = "/kaggle/input/datasets/rohithsheregar"
if not os.path.exists(INPUT_DIR):
    INPUT_DIR = "/kaggle/input/datasets" if os.path.exists("/kaggle/input/datasets") else "/kaggle/input"

print(f"📂 Input root: {INPUT_DIR}")
available_dirs = os.listdir(INPUT_DIR)
for d in available_dirs:
    print(f"   └─ {d}")

# ── Data symlinks (same as Module 1 Cell 2) ───────────────────────────────────
WORKING_DATA = "/kaggle/working/data"
if os.path.exists(WORKING_DATA):
    shutil.rmtree(WORKING_DATA)

MAPPINGS = {
    "train/oil":       lambda n: "oil" in n and not any(k in n for k in ["lookalike", "no", "test"]),
    "train/lookalike": lambda n: "lookalike" in n,
    "train/no_oil":    lambda n: "no" in n and "oil" in n,
    "test/oil":        lambda n: "test" in n,
}

total_tiffs = 0
print("\n🔗 Creating symlinks...")
for subpath, cond in MAPPINGS.items():
    matched = [d for d in available_dirs if cond(d.lower())]
    if not matched:
        print(f"   ⚠️  WARNING: no dataset matched for '{subpath}'")
        continue
    src = os.path.join(INPUT_DIR, matched[0])
    dst = os.path.join(WORKING_DATA, subpath)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    os.symlink(src, dst)
    n = len(glob.glob(os.path.join(src, "**", "*.tif*"), recursive=True))
    total_tiffs += n
    print(f"   ✅ {subpath} → {matched[0]}  ({n} TIFFs)")

print(f"\n📊 Total TIFFs: {total_tiffs}")

# ── Module 1 checkpoint (for on-the-fly inference) ───────────────────────────
M1_CKPT = None
CKPT_SEARCH_PATHS = [
    "/kaggle/working/results/module1/checkpoints/best_model.pt",
    "/kaggle/input/oil-spill-checkpoints/best_model.pt",
]
for p in CKPT_SEARCH_PATHS:
    if os.path.exists(p):
        M1_CKPT = p
        size_mb = os.path.getsize(p) / (1024**2)
        print(f"\n✅ Module 1 checkpoint: {p}  ({size_mb:.1f} MB)")
        break

if M1_CKPT is None:
    print("\nℹ️  No Module 1 checkpoint found — will use ground-truth masks directly.")
    print("   (This is fine if oil/lookalike gt masks are available in the dataset.)")

# ── Sanity check: discover scene pairs ───────────────────────────────────────
print("\n🔍 Dataset sanity check...")
from src.training.zenodo_sos_dataset import discover_sos_pairs

for cls in ["oil", "lookalike"]:
    cls_dir = Path(WORKING_DATA) / "train" / cls
    if cls_dir.exists():
        df = discover_sos_pairs(cls_dir, include_classes=[cls])
        print(f"   ✅ {cls:10s}: {len(df)} scene pairs found")
    else:
        print(f"   ⚠️  {cls:10s}: directory missing — {cls_dir}")

print("\n✅ Cell 2 complete.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 3 — FEATURE EXTRACTION (12-Feature Matrix)
# ─────────────────────────────────────────────────────────────────────────────
# Extracts exactly 12 tabular features per connected dark patch:
#
#   Polarimetric (4) : H, A (degenerate=0 for dual-pol), α, VV/VH ratio
#   Geometric    (4) : area_km², elongation, perimeter-to-area, compactness
#   Contextual   (3) : wind_speed_ms, proximity_shipping_lane_km, is_night
#   Temporal     (1) : morphology_change_km² (0.0 for single-pass)
#
# Source: Song et al. 2024, Chen & Wang 2022, Yang et al. 2022,
#         Liao et al. 2023, Li et al. 2023
# ─────────────────────────────────────────────────────────────────────────────
import time
import pandas as pd
import numpy as np
from pathlib import Path
import tifffile

from src.lookalike.morphology import close_and_extract
from src.lookalike.features   import (
    FEATURE_NAMES, META_COLUMNS,
    extract_scene_features, build_feature_dataframe,
)
from src.training.zenodo_sos_dataset import discover_sos_pairs

# ─── CONFIG ──────────────────────────────────────────────────────────────────
WORKING_DATA  = "/kaggle/working/data"
RESULTS_DIR   = "/kaggle/working/results/module2"
GSD_M         = 10.0       # Ground sampling distance — 10m for S1 IW GRD
MIN_COMP_PX   = 10         # Minimum patch size in pixels (= 1000 m² at 10m)
CLOSING_ITER  = 2          # Morphological closing iterations (Chang et al. 2024)
WIND_FALLBACK = 7.0        # ERA5 fallback m/s (open-ocean climatological mean)

import os
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/metrics", exist_ok=True)

print(f"📐 Feature matrix: {len(FEATURE_NAMES)} features")
print(f"   {FEATURE_NAMES}")
print(f"\n🔧 Config:")
print(f"   GSD           : {GSD_M} m")
print(f"   Min component : {MIN_COMP_PX} px")
print(f"   Closing iters : {CLOSING_ITER} (Chang et al. 2024)")
print(f"   Wind fallback : {WIND_FALLBACK} m/s")

# ─── Helper: load SAR arrays ─────────────────────────────────────────────────
def load_sar(image_path):
    try:
        arr = tifffile.imread(str(image_path)).astype(np.float32)
        if arr.ndim == 2:
            return arr, arr.copy()
        if arr.ndim == 3:
            if arr.shape[0] <= 8 and arr.shape[1] > 8:
                arr = np.moveaxis(arr, 0, -1)
            return arr[..., 0], arr[..., 1]
    except Exception as e:
        print(f"   ⚠️  Failed to load {image_path}: {e}")
    return None, None

def load_mask(mask_path):
    if mask_path is None or not Path(str(mask_path)).exists():
        return None
    try:
        mask = tifffile.imread(str(mask_path)).astype(np.uint8)
        if mask.ndim == 3:
            mask = mask[..., 0]
        return (mask > 0).astype(np.uint8)
    except Exception:
        return None

# ─── Process scenes ───────────────────────────────────────────────────────────
t0 = time.time()
all_feature_dfs = []
CLASS_LABELS    = {"oil": 1, "lookalike": 0}
scene_stats     = {"oil": 0, "lookalike": 0, "skipped": 0, "total_components": 0}

for cls_name, lbl in CLASS_LABELS.items():
    cls_dir = Path(WORKING_DATA) / "train" / cls_name
    if not cls_dir.exists():
        print(f"\n⚠️  {cls_name}: directory missing, skipping")
        continue

    df_pairs = discover_sos_pairs(cls_dir, include_classes=[cls_name])
    print(f"\n🔄 Processing {cls_name}: {len(df_pairs)} scenes (label={lbl})...")

    scene_dicts = []
    for idx, row in df_pairs.iterrows():
        vv, vh = load_sar(row["image_path"])
        if vv is None:
            scene_stats["skipped"] += 1
            continue

        # Load gt mask; if unavailable, use blank (inference mode)
        mask_path = row.get("mask_path", None)
        mask = load_mask(mask_path)
        if mask is None:
            # No mask available and no M1 inference here — skip
            scene_stats["skipped"] += 1
            continue

        # 2-iteration morphological closing + connected components
        _, regions = close_and_extract(
            binary_mask  = mask,
            iterations   = CLOSING_ITER,
            selem_size   = 5,
            min_area_px  = MIN_COMP_PX,
        )
        if not regions:
            continue

        label_map = {r.label: lbl for r in regions}
        scene_dicts.append({
            "scene_id":      row["scene_id"],
            "regions":       regions,
            "vv_db":         vv,
            "vh_db":         vh,
            "wind_speed_ms": WIND_FALLBACK,
            "hour_local":    12,         # Noon default; replace with scene metadata
            "label_map":     label_map,
        })

        if (idx + 1) % 100 == 0:
            print(f"   ... {idx+1}/{len(df_pairs)} scenes processed")

    feat_df = build_feature_dataframe(scene_dicts, gsd_m=GSD_M)
    scene_stats[cls_name] = len(feat_df)
    scene_stats["total_components"] += len(feat_df)
    all_feature_dfs.append(feat_df)
    print(f"   ✅ {cls_name}: {len(feat_df)} component rows extracted")

# ─── Combine and save ────────────────────────────────────────────────────────
ALL_FEATURES = pd.concat(all_feature_dfs, ignore_index=True) if all_feature_dfs else pd.DataFrame()

feat_csv = f"{RESULTS_DIR}/metrics/feature_summary.csv"
ALL_FEATURES.to_csv(feat_csv, index=False)

elapsed = time.time() - t0
print(f"\n{'='*60}")
print(f"📊 Feature Extraction Summary")
print(f"   Oil components      : {scene_stats['oil']}")
print(f"   Lookalike components: {scene_stats['lookalike']}")
print(f"   Skipped scenes      : {scene_stats['skipped']}")
print(f"   Total components    : {scene_stats['total_components']}")
print(f"   DataFrame shape     : {ALL_FEATURES.shape}")
print(f"   Saved to            : {feat_csv}")
print(f"   Elapsed             : {elapsed:.1f}s")
print(f"{'='*60}")
print(f"\n✅ Cell 3 complete.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4 — RANDOM FOREST TRAINING (GroupKFold CV by scene_id)
# ─────────────────────────────────────────────────────────────────────────────
# Trains the look-alike discriminator:
#   RF(n_estimators=200, class_weight='balanced_subsample')
#   grouped cross-validation: GroupKFold(groups=scene_id)
#   prevents data leakage — patches from same scene never cross train/val
#
# Source: Synopsis §2.1, Breiman (2001)
# ─────────────────────────────────────────────────────────────────────────────
import time, json
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src.lookalike.classifier  import LookalikeClassifier
from src.lookalike.features    import FEATURE_NAMES
from src.lookalike.bilge_filter import apply_bilge_filter, summarise_detections

# ─── CONFIG ──────────────────────────────────────────────────────────────────
N_ESTIMATORS   = 200     # Synopsis §2.1 — explicit requirement
N_FOLDS        = 5       # GroupKFold splits (capped to n_unique_scenes)
SEED           = 42
MIN_ELONGATION = 3.0     # Synopsis §2.2 — elongation > 3:1
MAX_AREA_KM2   = 50.0    # Synopsis §2.2 — area < 50 km²
NIGHT_BOOST    = 0.15    # Liao et al. 2023 — >80% illegal dumps at night
PROB_THRESHOLD = 0.50    # RF classification threshold after boost

print(f"🌲 RF Config:")
print(f"   n_estimators     : {N_ESTIMATORS}")
print(f"   class_weight     : 'balanced_subsample'  (per-tree, not global)")
print(f"   n_folds          : {N_FOLDS} GroupKFold by scene_id")
print(f"   Bilge elongation : > {MIN_ELONGATION}")
print(f"   Bilge area       : < {MAX_AREA_KM2} km²")
print(f"   Night boost      : +{NIGHT_BOOST} (Liao et al. 2023)")

# ─── Prepare training DataFrame ──────────────────────────────────────────────
if 'ALL_FEATURES' not in dir() or ALL_FEATURES.empty:
    print("\n❌ ALL_FEATURES not found — run Cell 3 first!")
    raise RuntimeError("Run Cell 3 (Feature Extraction) before Cell 4.")

train_df = ALL_FEATURES.dropna(subset=["label"]).copy()
train_df["label"] = train_df["label"].astype(int)

n_oil      = int((train_df["label"] == 1).sum())
n_lookalike= int((train_df["label"] == 0).sum())
n_scenes   = train_df["scene_id"].nunique()

print(f"\n📊 Training data:")
print(f"   Oil components      : {n_oil}")
print(f"   Lookalike components: {n_lookalike}")
print(f"   Unique scenes       : {n_scenes}  (GroupKFold groups)")
print(f"   Total rows          : {len(train_df)}")

if len(train_df) < 10:
    print("\n❌ Too few samples to train. Check Cell 3 output.")
    raise RuntimeError("Insufficient training data.")

# ─── Train ───────────────────────────────────────────────────────────────────
print(f"\n🚂 Training RandomForest (n_estimators={N_ESTIMATORS})...")
t0 = time.time()

clf = LookalikeClassifier(
    n_estimators = N_ESTIMATORS,
    n_folds      = N_FOLDS,
    random_state = SEED,
)
clf.fit(train_df, label_col="label", group_col="scene_id")

elapsed = time.time() - t0
print(f"\n✅ Training complete in {elapsed:.1f}s")

# ─── CV Results ───────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("📈 Cross-Validation Results (GroupKFold by scene_id)")
print(f"{'='*60}")
print(clf.cv_scores_.to_string(index=False))
mean_bacc = clf.cv_scores_["balanced_accuracy"].mean()
std_bacc  = clf.cv_scores_["balanced_accuracy"].std()
mean_auc  = clf.cv_scores_["auc"].mean()
std_auc   = clf.cv_scores_["auc"].std()
oob       = clf._rf.oob_score_ if hasattr(clf._rf, "oob_score_") else float("nan")
print(f"\n  Mean balanced_acc : {mean_bacc:.4f} ± {std_bacc:.4f}")
print(f"  Mean AUC          : {mean_auc:.4f} ± {std_auc:.4f}")
print(f"  OOB score         : {oob:.4f}")

# ─── Feature Importances ─────────────────────────────────────────────────────
fi_df = clf.feature_importance_df()
print(f"\n📊 Feature Importances (MDI — top 12):")
print(fi_df.to_string(index=False))

# ─── Feature importance plot ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
colors = ["#e63946" if imp > fi_df["importance"].mean() else "#457b9d"
          for imp in fi_df["importance"]]
ax.barh(fi_df["feature"], fi_df["importance"],
        xerr=fi_df["std"], color=colors,
        edgecolor="white", linewidth=0.5,
        error_kw={"elinewidth": 1.0, "capsize": 3, "ecolor": "#aaaaaa"})
ax.invert_yaxis()
ax.axvline(x=1/len(fi_df), color="grey", linestyle="--",
           linewidth=1.0, label=f"Uniform baseline (1/{len(fi_df)})")
ax.set_xlabel("Mean Decrease in Impurity (MDI)", fontsize=11)
ax.set_title("Module 2 — RF Feature Importances\n"
             f"(n_estimators={N_ESTIMATORS}, balanced_subsample)", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
fi_plot_path = f"{RESULTS_DIR}/metrics/feature_importance.png"
plt.savefig(fi_plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\n✅ Feature importance plot saved: {fi_plot_path}")

# ─── Save model ───────────────────────────────────────────────────────────────
model_path = f"{RESULTS_DIR}/checkpoints/lookalike_rf.joblib"
clf.save(model_path)
print(f"✅ Model saved: {model_path}")

# ─── Save CV scores ───────────────────────────────────────────────────────────
cv_path = f"{RESULTS_DIR}/metrics/cv_scores.json"
cv_records = clf.cv_scores_.to_dict(orient="records")
cv_records.append({
    "mean_balanced_accuracy": float(mean_bacc),
    "std_balanced_accuracy":  float(std_bacc),
    "mean_auc":               float(mean_auc),
    "std_auc":                float(std_auc),
    "oob_score":              float(oob),
})
import pathlib
pathlib.Path(cv_path).write_text(json.dumps(cv_records, indent=2))
print(f"✅ CV scores saved: {cv_path}")

# ─── Save feature importance CSV ──────────────────────────────────────────────
fi_csv = f"{RESULTS_DIR}/metrics/feature_importance.csv"
fi_df.to_csv(fi_csv, index=False)

# ─── Bilge-dump filter sanity check on training data ─────────────────────────
print(f"\n🔍 Applying bilge-dump filter (sanity check on training data)...")
proba_df  = clf.predict_proba(train_df)
train_aug = pd.concat([train_df.reset_index(drop=True), proba_df], axis=1)

filter_result = apply_bilge_filter(
    train_aug,
    prob_col        = "prob_oil",
    min_elongation  = MIN_ELONGATION,
    max_area_km2    = MAX_AREA_KM2,
    night_boost     = NIGHT_BOOST,
    prob_threshold  = PROB_THRESHOLD,
)
n_geom_pass = len(filter_result)
n_bilge     = int(filter_result["bilge_candidate"].sum())
print(f"   Geometry-passing patches : {n_geom_pass} / {len(train_aug)}")
print(f"   Bilge-dump candidates    : {n_bilge} / {n_geom_pass}")

det_summary = summarise_detections(filter_result)
det_path    = f"{RESULTS_DIR}/metrics/detection_summary.csv"
det_summary.to_csv(det_path, index=False)
print(f"   Detection summary saved  : {det_path}")

print(f"\n{'='*60}")
print("✅ Cell 4 complete.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5 — BILGE-DUMP FILTER RESULTS + SESSION SUMMARY + HF UPLOAD
# ─────────────────────────────────────────────────────────────────────────────
import os, json, shutil, glob, csv
from pathlib import Path
from datetime import datetime
import pandas as pd

RESULTS_DIR = "/kaggle/working/results/module2"
OUT_DIR     = "/kaggle/working/session_output_m2"
os.makedirs(OUT_DIR, exist_ok=True)

# ─── Collect output files ─────────────────────────────────────────────────────
print("📦 Collecting output files...")
collect_patterns = [
    f"{RESULTS_DIR}/checkpoints/*.joblib",
    f"{RESULTS_DIR}/metrics/*.csv",
    f"{RESULTS_DIR}/metrics/*.json",
    f"{RESULTS_DIR}/metrics/*.png",
]
for pat in collect_patterns:
    for src in glob.glob(pat):
        dst      = shutil.copy(src, OUT_DIR)
        size_mb  = os.path.getsize(dst) / (1024**2)
        print(f"   ✅ {Path(src).name}  ({size_mb:.2f} MB)")

print(f"\n📁 Output directory: {OUT_DIR}")

# ─── Print training summary ───────────────────────────────────────────────────
print(f"\n{'='*60}")
print("📊 Module 2 Training Summary")
print(f"{'='*60}")

cv_path = f"{RESULTS_DIR}/metrics/cv_scores.json"
if os.path.exists(cv_path):
    with open(cv_path) as f:
        cv = json.load(f)
    summary = cv[-1]  # last entry has the means
    print(f"   {'CV balanced_accuracy':<28}: {summary.get('mean_balanced_accuracy', float('nan')):.4f} ± {summary.get('std_balanced_accuracy', float('nan')):.4f}")
    print(f"   {'CV AUC':<28}: {summary.get('mean_auc', float('nan')):.4f} ± {summary.get('std_auc', float('nan')):.4f}")
    print(f"   {'OOB score':<28}: {summary.get('oob_score', float('nan')):.4f}")

det_path = f"{RESULTS_DIR}/metrics/detection_summary.csv"
if os.path.exists(det_path):
    det_df = pd.read_csv(det_path)
    print(f"\n   {'Total scenes analysed':<28}: {len(det_df)}")
    print(f"   {'Total bilge candidates':<28}: {det_df['n_bilge_dumps'].sum()}")
    print(f"   {'Scenes with detections':<28}: {(det_df['n_bilge_dumps'] > 0).sum()}")

fi_path = f"{RESULTS_DIR}/metrics/feature_importance.csv"
if os.path.exists(fi_path):
    fi_df = pd.read_csv(fi_path)
    print(f"\n   Top-3 features by importance:")
    for _, r in fi_df.head(3).iterrows():
        print(f"     {r['feature']:<35}: {r['importance']:.4f} ± {r['std']:.4f}")

# ─── Write train_metrics.csv ─────────────────────────────────────────────────
metrics_row = {
    "module":              "module2",
    "n_estimators":        N_ESTIMATORS if 'N_ESTIMATORS' in dir() else 200,
    "mean_cv_bacc":        summary.get("mean_balanced_accuracy", float("nan")) if os.path.exists(cv_path) else float("nan"),
    "mean_cv_auc":         summary.get("mean_auc", float("nan")) if os.path.exists(cv_path) else float("nan"),
    "oob_score":           summary.get("oob_score", float("nan")) if os.path.exists(cv_path) else float("nan"),
    "timestamp":           datetime.now().isoformat(),
}
train_metrics_path = f"{RESULTS_DIR}/metrics/train_metrics.csv"
with open(train_metrics_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(metrics_row.keys()))
    w.writeheader()
    w.writerow(metrics_row)
print(f"\n✅ Metrics CSV: {train_metrics_path}")

# ─── Upload to Hugging Face Hub ───────────────────────────────────────────────
UPLOAD_FILES = [
    f"{RESULTS_DIR}/checkpoints/lookalike_rf.joblib",
    f"{RESULTS_DIR}/metrics/cv_scores.json",
    f"{RESULTS_DIR}/metrics/feature_importance.png",
    f"{RESULTS_DIR}/metrics/feature_importance.csv",
    f"{RESULTS_DIR}/metrics/detection_summary.csv",
    f"{RESULTS_DIR}/metrics/train_metrics.csv",
]

if HF_TOKEN and HF_REPO_ID:
    print(f"\n🤗 Uploading to Hugging Face Hub ({HF_REPO_ID})...")
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    for fpath in UPLOAD_FILES:
        if os.path.exists(fpath):
            name    = Path(fpath).name
            size_mb = os.path.getsize(fpath) / (1024**2)
            try:
                api.upload_file(
                    path_or_fileobj=fpath,
                    path_in_repo=f"module2/{name}",
                    repo_id=HF_REPO_ID,
                    commit_message=f"Module2 auto-save: {name}",
                )
                print(f"   ✅ Uploaded: module2/{name}  ({size_mb:.2f} MB)")
            except Exception as e:
                print(f"   ⚠️  Upload failed for {name}: {e}")
    print(f"\n   🔗 https://huggingface.co/{HF_REPO_ID}")
else:
    print("\nℹ️  HF upload skipped (no token / repo configured).")

print(f"\n{'='*60}")
print("✅ Module 2 training complete!")
print(f"   Model     : {RESULTS_DIR}/checkpoints/lookalike_rf.joblib")
print(f"   Plots     : {RESULTS_DIR}/metrics/feature_importance.png")
print(f"   Output tab: {OUT_DIR}")
print(f"{'='*60}")

---
## 📋 Module 2 Quick Reference

### What Module 2 does
1. **Morphological closing** (2-iter, 5×5 square — Chang et al. 2024) fills SAR segmentation fragments
2. **12-feature extraction** per connected dark patch:
   - *Polarimetric:* H, A (=0 dual-pol), α, VV/VH ratio (Song 2024, Chen 2022)
   - *Geometric:* area km², elongation, perimeter-area ratio, compactness (Yang 2022)
   - *Contextual:* wind speed, shipping-lane proximity, is_night (Liao 2023)
   - *Temporal:* morphology change km² (Li 2023)
3. **Random Forest (n=200, balanced_subsample)** trained with GroupKFold by scene_id
4. **Bilge-dump filter:** elongation > 3:1 AND area < 50 km² AND night-time boost +0.15

### Output files
| File | Purpose |
|------|---------|
| `checkpoints/lookalike_rf.joblib` | Trained RF classifier |
| `metrics/cv_scores.json` | Per-fold balanced_accuracy + AUC |
| `metrics/feature_importance.png` | Bar chart of MDI importances |
| `metrics/feature_importance.csv` | Ranked feature importances |
| `metrics/detection_summary.csv` | Per-scene bilge candidate counts |
| `metrics/train_metrics.csv` | Module-level summary row |

### Run all cells in order:
**Cell 0 → Cell 1 → Cell 2 → Cell 3 → Cell 4 → Cell 5**

Expected total runtime: **< 1 hour** on T4 GPU (mostly feature extraction, RF trains in minutes)